# 07 Player Case Study Generator

## Coach's question

How can analysts generate reproducible player case studies with explicit filters and review IDs?

## Match situation

Parameter-driven player/team/competition/phase/action/reliability filtering for coach-ready event queues.

## What the current data can measure

- Event-level defensive actions
- OOF expected threat and observed short-horizon outcomes
- Sequence windows from full processed event timeline
- Visibility and 360 reliability coverage

## What it cannot measure

- Body orientation and exact scanning behaviour
- Off-camera positioning beyond freeze-frame visibility
- Full tactical intent without video review

## Data population and filters

Population, reliability filters, and model variants are computed in the code cell and reported in a notebook summary dictionary.

## Descriptive evidence

Action mix, zone mix, and continuation outcomes are exported to `outputs/coach_analysis/tables/`.

## Model-based evidence

R4 and two-part expected xG are compared with observed outcomes and suppression fields.

## Tactical interpretation

Rules are evidence-based and any ambiguous cases are tagged for video review.

## Representative situations for video review

Category-specific representative events are exported with explicit selection reason.

## Coaching takeaways

The notebook prints a direct coach-facing conclusion with sample sizes and uncertainty flags.

## Limitations

Case studies are descriptive and should be validated with match video before coaching decisions.

## Follow-up question

Which custom parameter combinations best separate stable strengths from high-variance episodes?

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'requirements.txt').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dax.coach_analysis.notebook_utils import (
    notebook_00_readiness_summary,
    notebook_rule_samples,
    prepare_coach_frame,
    select_population,
    write_notebook_outputs,
)

NOTEBOOK_ID = '07'
NOTEBOOK_SLUG = '07_player_case_study_generator'
frame, events, coverage = prepare_coach_frame(ROOT)
population = select_population(frame, NOTEBOOK_ID)


PLAYER = None
TEAM = None
COMPETITION = None
PHASE = None
ACTION_FAMILY = None
RELIABLE_ONLY = False
if PLAYER:
    population = population[population['player'].eq(PLAYER)]
if TEAM:
    population = population[population['team'].eq(TEAM)]
if COMPETITION:
    population = population[population['competition_label'].eq(COMPETITION)]
if PHASE:
    population = population[population['phase_label'].eq(PHASE)]
if ACTION_FAMILY:
    population = population[population['action_family'].eq(ACTION_FAMILY)]
if RELIABLE_ONLY and 'coach_reliable_visibility' in population.columns:
    population = population[population['coach_reliable_visibility'].eq(True)]


summary = write_notebook_outputs(
    NOTEBOOK_SLUG,
    population,
    groupings=[['player'], ['team'], ['competition_label'], ['phase_label'], ['action_family']],
    video_n=3,
    include_labels=NOTEBOOK_ID in {'01', '05'},
)
summary['oof_alignment'] = coverage
summary['conclusion'] = 'Case-study filters produce auditable player event sets with explicit review reasons and sample-size flags.'
if NOTEBOOK_ID == '00':
    summary['schema_inventory'] = notebook_00_readiness_summary().to_dict(orient='records')
if NOTEBOOK_ID in {'01', '05'}:
    summary['label_rule_samples'] = notebook_rule_samples(population).to_dict(orient='records')

print(summary)